In [ ]:
import asyncio
import nodriver as uc
import random
import traceback
import re

TARGET_PAGE_URL = "https://www.facebook.com/mocseafood/"

async def inspect_html_and_click():
    print("🚀 Mở trình duyệt chống chống Bot...")
    browser = await uc.start(headless=False, icon_mode=1)

    try:
        # ========== BƯỚC 1: VÀO GOOGLE SEARCH ==========
        print("🔍 Vào Google tìm kiếm...")
        page = await browser.get("https://www.google.com/search?q=site:facebook.com/mocseafood")

        print("⏳ Chờ trang tải hoàn thiện (6s)...")
        await asyncio.sleep(6)

        # ========== BƯỚC 2: LẤY HTML THẬT TỪ DOM ĐÃ RENDER ==========
        print("📸 Đang lấy toàn bộ nội dung HTML thật từ DOM (live) của trình duyệt...")
        html_content = await page.get_content()

        with open("google_real_inspect.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        print("💾 Đã lưu HTML thật vào file 'google_real_inspect.html' để làm bằng chứng.")

        # ========== BƯỚC 3: TÌM ĐÚNG ELEMENT TRANG CHỦ TRÊN UI ==========
        # Lưu ý: class="zReHs" và jsname="UWckNb" là thuộc tính CHUNG của Google
        # cho MỌI link kết quả tự nhiên (cả link trang chủ và link ảnh đều có).
        # => Không thể dùng riêng class/jsname để phân biệt, phải khóa thêm href tuyệt đối.
        print(f"🎯 Đang tìm chính xác link trang chủ: {TARGET_PAGE_URL}")

        # Ưu tiên 1: selector kết hợp class + jsname + href tuyệt đối (chuẩn nhất)
        selector_strict = f'a.zReHs[jsname="UWckNb"][href="{TARGET_PAGE_URL}"]'
        target_element = await page.select(selector_strict)

        # Ưu tiên 2: nếu Google đổi class/jsname, fallback chỉ dùng href tuyệt đối
        if not target_element:
            print("⚠️ Selector class+jsname không khớp (có thể Google đã đổi class), thử fallback href tuyệt đối...")
            target_element = await page.select(f'a[href="{TARGET_PAGE_URL}"]')

        # Ưu tiên 3: fallback bằng XPath nếu CSS selector vẫn không ăn
        if not target_element:
            print("⚠️ Tìm bằng CSS selector hụt, thử tìm bằng XPath tuyệt đối...")
            result = await page.xpath(f'//a[@href="{TARGET_PAGE_URL}"]')
            if isinstance(result, list) and len(result) > 0:
                target_element = result[0]
            else:
                target_element = result

        if not target_element:
            print("❌ Không tìm thấy link trang chủ chính xác trên trang kết quả.")
            return

        print("✅ Đã tìm thấy đúng element link trang chủ!")
        print(f"   ↳ Tiêu đề nút: {target_element.text.strip()[:50] if target_element.text else 'Trống'}")

        # ========== BƯỚC 4: MÔ PHỎNG VÀ CLICK ==========
        delay = random.uniform(1.5, 2.5)
        print(f"⏳ Nghỉ tay {delay:.2f}s cho mượt...")
        await asyncio.sleep(delay)

        print("🖱️ Thực hiện Click vào link trang chủ...")
        await target_element.click()
        print("  → Đã phát lệnh Click thành công!")

        # ========== BƯỚC 5: CHỜ TRANG ĐÍCH LOAD ==========
        print("⏳ Chờ trang đích điều hướng (10s)...")
        await asyncio.sleep(10)

        print(f"📍 URL hiện tại: {page.url}")
        if "facebook.com/mocseafood" in page.url.lower():
            print("🎉 XUẤT SẮC: Đã vào đúng trang chủ Facebook!")
        else:
            print("⚠️ URL chưa chuyển đổi đúng như mong đợi. Hãy kiểm tra màn hình thực tế.")

    except Exception as e:
        print(f"❌ Đã xảy ra lỗi hệ thống: {e}")
        traceback.print_exc()

    finally:
        print("\n🔒 Đang giữ trình duyệt 5s để quan sát trước khi tắt...")
        await asyncio.sleep(5)
        try:
            browser.stop()
        except Exception:
            pass
        print("🛑 Đã đóng trình duyệt sạch sẽ.")

if __name__ == "__main__":
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(inspect_html_and_click())

mẫu 2 tốt hơn

In [ ]:
import asyncio
import random
import traceback

import nodriver as uc

TARGET_PAGE_URL = "https://www.facebook.com/mocseafood/"


async def inspect_html_and_click():
    browser = None

    try:
        print("🚀 Mở trình duyệt...")
        browser = await uc.start(
            headless=False,
            icon_mode=1
        )

        print("🔍 Vào Google...")
        page = await browser.get(
            "https://www.google.com/search?q=site:facebook.com/mocseafood"
        )

        print("⏳ Đợi kết quả xuất hiện...")

        target_element = None

        for _ in range(30):
            target_element = await page.select(
                f'a[href="{TARGET_PAGE_URL}"]'
            )

            if target_element:
                break

            await asyncio.sleep(0.2)

        if not target_element:
            print("❌ Không tìm thấy link.")
            return

        print("✅ Đã tìm thấy link.")

        try:
            await target_element.scroll_into_view()
        except Exception:
            pass

        await asyncio.sleep(
            random.uniform(0.3, 0.8)
        )

        print("🖱️ Click...")

        old_tab_count = len(browser.tabs)

        await target_element.click()

        await asyncio.sleep(
            random.uniform(1.0, 2.0)
        )

        print("🔎 Kiểm tra tabs...")

        found = False

        for tab in browser.tabs:
            try:
                url = str(tab.url)

                print("TAB:", url)

                if "facebook.com/mocseafood" in url.lower():
                    found = True
                    print("🎉 Đã mở đúng Facebook.")
                    break

            except Exception:
                pass

        if not found:
            print("⚠️ Không tìm thấy tab Facebook.")

    except Exception as e:
        print("❌ Lỗi:", e)
        traceback.print_exc()

    finally:
        print("⏳ Giữ trình duyệt 30s...")
        await asyncio.sleep(30)

        try:
            if browser:
                browser.stop()
        except Exception:
            pass


if __name__ == "__main__":
    await inspect_html_and_click()